# Notebook 01 — Data Loading & QA Dataset (Kaggle T4)

Runs on Kaggle T4 with 30 hours compute. Downloads Indiana University CXR dataset and generates QA pairs via Groq.

**Before running:** Add Kaggle Secrets:
- `GROQ_API_KEY`
- `KAGGLE_USERNAME`, `KAGGLE_KEY` (for dataset download)
- `HF_TOKEN`

In [ ]:
!pip install -q groq tqdm pandas kaggle

In [ ]:
import os, sys
import subprocess

WORKING_DIR = '/kaggle/working'
os.makedirs(WORKING_DIR, exist_ok=True)

# Kaggle secrets are auto-loaded into environment
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
HF_TOKEN = os.environ.get('HF_TOKEN')
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME')
KAGGLE_KEY = os.environ.get('KAGGLE_KEY')

if not all([GROQ_API_KEY, HF_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY]):
    print('ERROR: Missing Kaggle secrets. Add to notebook secrets.')
    sys.exit(1)

print('✓ Secrets loaded')

In [ ]:
# Clone repo if needed
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')
if not os.path.exists(REPO_PATH):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    print('Updating repository...')
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)
print('✓ Repository ready')

In [ ]:
# Download Indiana University CXR dataset
KAGGLE_DIR = os.path.join(WORKING_DIR, 'openi')

if not os.path.exists(os.path.join(KAGGLE_DIR, 'indiana_reports.csv')):
    print('Downloading dataset (~1 GB)...')
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'raddar/chest-xrays-indiana-university',
        '-p', KAGGLE_DIR,
        '--unzip'
    ], check=True)
    print('✓ Dataset downloaded')
else:
    print('✓ Dataset already present')

print(f'\nContents of {KAGGLE_DIR}:')
for f in os.listdir(KAGGLE_DIR):
    print(f'  {f}')

In [ ]:
# Auto-detect image directory
import glob

png_files = glob.glob(os.path.join(KAGGLE_DIR, '**', '*.png'), recursive=True)
if not png_files:
    raise RuntimeError(f'No PNG images found in {KAGGLE_DIR}')

IMAGES_DIR = os.path.dirname(png_files[0])
print(f'Found {len(png_files)} PNG images in: {IMAGES_DIR}')

In [ ]:
# Load dataset from Kaggle CSVs
from src.data.openi_loader import OpenILoader

loader = OpenILoader(images_dir=IMAGES_DIR)
df = loader.load_from_kaggle_csvs(kaggle_dir=KAGGLE_DIR)

print(f'Loaded {len(df)} studies with impression + frontal image')
print(f'Columns: {df.columns.tolist()}')
print(df.head(3))

In [ ]:
# Train / val / test split
import pandas as pd

train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

# Save to working directory
os.makedirs(os.path.join(REPO_PATH, 'data', 'processed'), exist_ok=True)
corpus_path = os.path.join(REPO_PATH, 'data', 'processed', 'reports_corpus.csv')
full_df.to_csv(corpus_path, index=False)

# Also save to working dir for easy access
full_df.to_csv(os.path.join(WORKING_DIR, 'reports_corpus.csv'), index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Sample impressions:')
for imp in full_df['impression'].head(3):
    print(f'  {imp[:120]}')

In [ ]:
# Configure Groq
from src.data.qa_creator import QACreator

creator = QACreator(groq_api_key=GROQ_API_KEY)
print('QA creator ready')

In [ ]:
# Generate QA dataset (max_studies=200 for quick run, or None for full)
QA_OUTPUT = os.path.join(WORKING_DIR, 'qa_dataset.jsonl')

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=200,  # Change to None for full dataset (~3 hours)
)

print(f'Generated {len(pairs)} QA pairs')
print(f'Saved to: {QA_OUTPUT}')

In [ ]:
# Inspect sample QA pairs
import json

with open(QA_OUTPUT) as f:
    samples = [json.loads(l) for l in f][:5]

for s in samples:
    print(f"Category : {s['category']}")
    print(f"Q        : {s['question']}")
    print(f"A        : {s['answer']}")
    print()

In [ ]:
# Dataset statistics
qa_df = pd.read_json(QA_OUTPUT, lines=True)
print(f'Total pairs   : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'\nSplit distribution:')
print(qa_df['split'].value_counts())
print(f'\nCategory distribution:')
print(qa_df['category'].value_counts())

print(f'\n✓ All data saved to {WORKING_DIR}')